# Repair `token_shards_merged/` from the cold-backup tar (sequential variant)

Targeted repair without random access into Drive. Steps:

1. Diff `token_shards_merged/` against the merged manifest -> list of missing merged shards.
2. Walk each missing shard's `merged_from` -> distinct small shards we need.
3. **One sequential pass** of `cat token_shards.tar.part-* | tar -x --files-from=needed.txt`, extracting those small shards into `/content/small_shards/` (local SSD).
4. For each missing merged shard, concatenate its sources from `/content/small_shards/` (in `merged_from` order) and write the result back to `token_shards_merged/` on Drive.
5. Validate.

Why this shape: random byte-offset reads across a 306 GB archive on Drive FUSE thrash the local cache and fill `/content`. Sequential streaming evicts as it reads, so it stays bounded. Drive writes are limited to the few hundred MB of repaired merged shards.

**If you are restarting after a failed run, restart the Colab runtime first** to clear `/content` (Runtime → Restart runtime). The previous failed run probably left tens of GB of FUSE cache behind.

In [ ]:
# 1. Mount Drive, verify parts, load merged manifest, check local free space.
from google.colab import drive
drive.mount('/content/drive')

import os, glob, json, shutil

SYNAPSE          = '/content/drive/MyDrive/synapse'
TARS_DIR         = os.path.join(SYNAPSE, 'token_shards')
MERGED_DIR       = os.path.join(SYNAPSE, 'token_shards_merged')
LOCAL_SCRATCH    = '/content/small_shards'
LOCAL_LIST       = '/content/needed_small.txt'
LOCAL_LOG        = '/content/tar_extract.log'
os.makedirs(LOCAL_SCRATCH, exist_ok=True)

parts = sorted(glob.glob(os.path.join(TARS_DIR, 'token_shards.tar.part-*')))
expected_suffixes = ['aa', 'ab', 'ac', 'ad', 'ae', 'af']
got_suffixes = [os.path.basename(p).rsplit('part-', 1)[-1] for p in parts]
if got_suffixes != expected_suffixes:
    raise FileNotFoundError(f'Expected {expected_suffixes}, got {got_suffixes}')
for p in parts:
    if os.path.getsize(p) == 0:
        raise RuntimeError(f'{os.path.basename(p)} is zero bytes')

tar_total = sum(os.path.getsize(p) for p in parts)
print(f'Tar parts: {len(parts)}, total {tar_total/1e9:.1f} GB')

with open(os.path.join(MERGED_DIR, 'shard_manifest.json')) as f:
    merged_manifest = json.load(f)
with open(os.path.join(MERGED_DIR, 'meta.json')) as f:
    merged_meta = json.load(f)
DTYPE_BYTES = {'uint8':1,'uint16':2,'uint32':4,'int8':1,'int16':2,'int32':4}[
    merged_meta.get('shard_dtype', 'uint16')]
print(f'Merged manifest: {len(merged_manifest["shards"])} shards, {DTYPE_BYTES}B/token')

free_gb = shutil.disk_usage('/content').free / 1e9
print(f'/content free: {free_gb:.1f} GB')
if free_gb < 60:
    raise RuntimeError(
        f'/content has only {free_gb:.1f} GB free. The previous indexing run probably '
        f'left FUSE cache behind. **Restart the Colab runtime** (Runtime -> Restart runtime), '
        f'then re-run this notebook.')

In [ ]:
# 2. Identify missing merged shards and resolve to needed small shards.
expected_merged = {s['shard']: s for s in merged_manifest['shards']}
present_merged  = {os.path.basename(p) for p in glob.glob(os.path.join(MERGED_DIR, 'shard_*.bin'))}
missing_merged  = sorted(set(expected_merged) - present_merged)

print(f'Merged shards: expected={len(expected_merged)} present={len(present_merged)} missing={len(missing_merged)}')
if not missing_merged:
    print('Nothing to repair. Re-run training.')
    raise SystemExit

needed_small = {}            # small shard name -> bytes expected
missing_specs = []           # one per missing merged shard
for name in missing_merged:
    entry = expected_merged[name]
    sources = entry.get('merged_from')
    if not sources:
        raise RuntimeError(f'{name} has no merged_from in manifest — cannot reconstruct.')
    spec = {
        'merged_shard': name,
        'sources': [{'shard': s['shard'], 'tokens': s['tokens']} for s in sources],
        'expected_bytes': sum(s['tokens'] * DTYPE_BYTES for s in sources),
    }
    missing_specs.append(spec)
    for s in sources:
        needed_small[s['shard']] = s['tokens'] * DTYPE_BYTES

needed_total_gb = sum(needed_small.values()) / 1e9
print(f'Distinct small shards needed: {len(needed_small)}  ({needed_total_gb:.2f} GB)')
if needed_total_gb > free_gb - 10:
    raise RuntimeError(
        f'Need ~{needed_total_gb:.1f} GB of small shards locally, but only '
        f'{free_gb:.1f} GB free on /content. Either free space, run on a VM with more disk, '
        f'or process in batches (split missing_specs across multiple notebook runs).')

In [ ]:
# 3. Detect the tar member name format (e.g. './data_code/shard_00012.bin' vs
#    'data_code/shard_00012.bin') by reading the first regular file header out
#    of part-aa. Build a --files-from list with names matching that format.
def detect_first_file_name(part_path, max_headers=20):
    with open(part_path, 'rb') as f:
        for _ in range(max_headers):
            head = f.read(512)
            if len(head) < 512:
                return None
            name = head[:100].rstrip(b'\x00 ').decode(errors='replace')
            type_flag = head[156:157]
            magic = head[257:265]
            if b'ustar' not in magic:
                raise RuntimeError('part-aa: invalid tar header magic — file is corrupt.')
            try:
                size = int(head[124:136].rstrip(b' \x00').decode() or '0', 8)
            except ValueError:
                return None
            if type_flag in (b'0', b'\x00') and name:
                return name
            f.seek(((size + 511) // 512) * 512, 1)   # skip member data
    return None

first_file = detect_first_file_name(parts[0])
print(f'First file member in tar: {first_file!r}')
if not first_file:
    raise RuntimeError('Could not find a regular-file header in part-aa.')

tar_prefix = './' if first_file.startswith('./') else ''
print(f'Tar member prefix: {tar_prefix!r}')

def to_tar_name(s_shard):
    return tar_prefix + s_shard.replace('\\', '/')

with open(LOCAL_LIST, 'w') as f:
    for n in sorted(needed_small):
        f.write(to_tar_name(n) + '\n')
print(f'Wrote {len(needed_small)} names -> {LOCAL_LIST}')
print('First 3 entries:')
with open(LOCAL_LIST) as f:
    for line in [next(f) for _ in range(3)]:
        print(f'  {line.rstrip()}')

In [ ]:
# 4. Sequential extraction. Streams 306 GB of tar in one pass, but only writes
#    the small shards listed in needed_small.txt. Writes to /content (local SSD).
#    Long-running (~1 hour bound by Drive read speed). Tails progress live.
import subprocess, time

if os.path.exists(LOCAL_LOG):
    os.remove(LOCAL_LOG)

parts_glob = os.path.join(TARS_DIR, 'token_shards.tar.part-*')
cmd = (
    f"set -o pipefail; "
    f"cat {parts_glob} | "
    f"tar -xf - -C '{LOCAL_SCRATCH}' --files-from='{LOCAL_LIST}' "
    f"--no-overwrite-dir --skip-old-files "
    f"> '{LOCAL_LOG}' 2>&1"
)
print('Starting sequential extraction. Expect ~1 hour.')
t0 = time.time()
proc = subprocess.Popen(cmd, shell=True, executable='/bin/bash')

try:
    while proc.poll() is None:
        time.sleep(20)
        extracted = sum(
            1 for _ in glob.iglob(os.path.join(LOCAL_SCRATCH, '**', '*.bin'), recursive=True)
        )
        free_now_gb = shutil.disk_usage('/content').free / 1e9
        elapsed_min = (time.time() - t0) / 60
        print(f'[{elapsed_min:6.1f} min]  extracted={extracted}/{len(needed_small)}  '
              f'/content_free={free_now_gb:.1f} GB')
        if free_now_gb < 5:
            print('WARNING: /content nearly full — killing tar to avoid corruption.')
            proc.terminate()
            break
except KeyboardInterrupt:
    proc.terminate()
    raise

rc = proc.wait()
print(f'\ntar exit code: {rc}  ({(time.time()-t0)/60:.1f} min)')
print('\nLast 30 log lines:')
print(subprocess.check_output(['tail', '-30', LOCAL_LOG], text=True))
# tar exit codes: 0=ok, 1=some warnings (e.g. file changed), 2=fatal
if rc not in (0, 1):
    print(f'WARNING: tar exited with {rc}. Continuing — cell 5 will check what is missing.')

In [ ]:
# 5. Verify all needed small shards landed locally with correct sizes.
got_small = {}        # basename -> local path
for p in glob.iglob(os.path.join(LOCAL_SCRATCH, '**', '*.bin'), recursive=True):
    got_small[os.path.basename(p)] = p

missing_local = []
size_bad = []
for n, expected_bytes in needed_small.items():
    base = os.path.basename(n)
    p = got_small.get(base)
    if p is None:
        missing_local.append(n)
        continue
    sz = os.path.getsize(p)
    if sz != expected_bytes:
        size_bad.append((n, expected_bytes, sz))

print(f'Needed small shards:    {len(needed_small)}')
print(f'Extracted locally:      {len(got_small)}')
print(f'Missing after extract:  {len(missing_local)}')
print(f'Wrong-sized:            {len(size_bad)}')
if missing_local:
    print(f'  First 10 missing: {missing_local[:10]}')
    raise RuntimeError(
        f'{len(missing_local)} small shard(s) were not in the tar (or tar bailed before reaching them). '
        f'If the tar bailed early, the corrupt part is the one containing the first missing entry.')
if size_bad:
    for n, exp, got in size_bad[:5]:
        print(f'  {n}: expected {exp}B, got {got}B')
    raise RuntimeError('Size mismatches between tar contents and merged manifest — tar is corrupt or stale.')
print('\nAll required small shards present and correctly sized.')

In [ ]:
# 6. Merge: for each missing merged shard, concat its sources from /content into
#    a temp file on Drive, verify size, atomic-rename to the final .bin.
import time
COPY_CHUNK = 4 * 1024 * 1024

repaired = 0
failed = []
for i, spec in enumerate(missing_specs, 1):
    out_path = os.path.join(MERGED_DIR, spec['merged_shard'])
    tmp_path = out_path + '.tmp'
    t0 = time.time()
    try:
        with open(tmp_path, 'wb') as fout:
            for s in spec['sources']:
                src_path = got_small[os.path.basename(s['shard'])]
                with open(src_path, 'rb') as fin:
                    while True:
                        chunk = fin.read(COPY_CHUNK)
                        if not chunk: break
                        fout.write(chunk)
        actual = os.path.getsize(tmp_path)
        if actual != spec['expected_bytes']:
            raise RuntimeError(f'byte mismatch: wrote {actual}B, expected {spec["expected_bytes"]}B')
        os.replace(tmp_path, out_path)
        repaired += 1
        print(f'  [{i}/{len(missing_specs)}] {spec["merged_shard"]}: '
              f'{len(spec["sources"])} src, {actual/1e6:.1f} MB, {time.time()-t0:.1f}s')
    except Exception as e:
        failed.append((spec['merged_shard'], str(e)))
        if os.path.exists(tmp_path): os.remove(tmp_path)
        print(f'  [{i}/{len(missing_specs)}] {spec["merged_shard"]}: FAILED — {e}')

print(f'\nRepaired: {repaired}/{len(missing_specs)}')
if failed:
    raise RuntimeError(f'{len(failed)} merged shard(s) could not be repaired:\n  ' +
                       '\n  '.join(f'{n}: {e}' for n, e in failed[:20]))

In [ ]:
# 7. Final validation against merged manifest.
present = {os.path.basename(p) for p in glob.glob(os.path.join(MERGED_DIR, 'shard_*.bin'))}
still_missing = sorted(set(expected_merged) - present)
extra         = sorted(present - set(expected_merged))
print(f'Expected: {len(expected_merged)}  Present: {len(present)}  Missing: {len(still_missing)}  Extra: {len(extra)}')
if still_missing:
    print(f'  first 10 missing: {still_missing[:10]}')
    raise RuntimeError(f'{len(still_missing)} merged shard(s) still missing.')
if extra:
    print(f'  extras (not in manifest, leave alone or audit): {extra[:5]}')
print('\ntoken_shards_merged/ matches the merged manifest. Re-run training.')

In [ ]:
# 8. Cleanup local scratch (free /content for the rest of your session).
import shutil as _shutil
if os.path.exists(LOCAL_SCRATCH):
    _shutil.rmtree(LOCAL_SCRATCH)
for p in (LOCAL_LIST, LOCAL_LOG):
    if os.path.exists(p): os.remove(p)
free_gb = _shutil.disk_usage('/content').free / 1e9
print(f'Cleaned up. /content free: {free_gb:.1f} GB')